In [ ]:
import numpy as np
import openpmd_viewer as ioview
%matplotlib widget
import scipy.constants as sc
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
ts = ioview.OpenPMDTimeSeries("./diags/fields/")

In [ ]:
20*np.log10(7.484578e-04)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from ipywidgets import interact, IntSlider
from openpmd_viewer import OpenPMDTimeSeries

ts = OpenPMDTimeSeries("./diags/fields/")

def plot_iteration(iteration_index):
    iteration = ts.iterations[iteration_index]
    
    # Load your field of interest
    Ex, info = ts.get_field(field='E', coord='y', slice_across='y', iteration=iteration)
    
    # Load eb_covered mask (1 = covered/metal, 0 = vacuum)
    eb, _ = ts.get_field(field='eb_covered', coord=None, slice_across='y', iteration=iteration)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Field plot
    im = ax.imshow(
        Ex,
        extent=info.imshow_extent,
        origin='lower',
        aspect=1.0,
        cmap='RdBu',
        vmin=-1e5,
        vmax=1e5,
    )
    plt.colorbar(im, ax=ax, label='Ey (V/m)')
    
    # eb_covered on top: 1=black (metal), 0=transparent (vacuum)
    # Build a colormap: 0 -> fully transparent, 1 -> black
    cmap_eb = mcolors.LinearSegmentedColormap.from_list(
        'eb_mask', [(0, 0, 0, 0), (0, 0, 0, 1)]  # (R,G,B,A) tuples
    )
    ax.imshow(
        eb,
        extent=info.imshow_extent,
        origin='lower',
        aspect=1.0,
        cmap=cmap_eb,
        vmin=0.0,
        vmax=1.0,
        interpolation='nearest',   # sharp edges on the geometry
    )
    
    ax.set_title(f'Iteration {iteration}')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    plt.tight_layout()
    plt.show()

interact(
    plot_iteration,
    iteration_index=IntSlider(min=0, max=len(ts.iterations)-1, step=1, value=0,
                              description='Iteration')
)

In [ ]:
ts.slider(cmap='RdBu_r',vmin=-1e6,vmax=1e6)

In [ ]:
FE = pd.read_csv("./diags/reducedfiles/field_energy.txt", delimiter="\s+")

In [ ]:
FE

In [ ]:
FE.plot(x="[1]time(s)", y="[2]total_lev0(J)",logy=True)

In [ ]:
# The zero-th energy value is zero
time_s = FE["[1]time(s)"][1:]
energy_J = FE["[2]total_lev0(J)"][1:]

energy_dB = 10*np.log10(energy_J/np.max(energy_J))

In [ ]:
ns = 1e-9

fig,ax = plt.subplots(1,1)

ax.plot(time_s / ns, energy_dB)
ax.set_xlabel("Time (ns)")
ax.set_ylabel("EM energy in system (dB)")

In [ ]:
s21_fft = np.load("./analysis/s21.npz",)

In [ ]:
s21_fft

In [ ]:
f0 = 67e9  # base frequency

In [ ]:
fig,ax = plt.subplots(1,1)

ax.plot(s21_fft["freqs"]/1e9,20*np.log10(np.abs(s21_fft["S21"])))
ax.axvline(x=f0/1e9, c="r", ls="--")

In [ ]:
data = np.load("./analysis/s21.npz", allow_pickle=True)

# --- FFT result (spectrum) ---
freqs = data["freqs"]
S21   = data["S21"]

# Wrapped phase: values stay in (-180, 180] — standard for S-params
phase_wrapped = np.degrees(np.angle(S21))

# Unwrapped phase: continuous, removes 2π jumps — useful to see
# total phase accumulation across frequency
phase_unwrapped = np.degrees(np.unwrap(np.angle(S21)))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

ax = axes[0]
ax.plot(freqs * 1e-9, 20 * np.log10(np.abs(S21) + 1e-30))
ax.set_ylabel("|S21| (dB)")
ax.axvline(67, color="r", ls="--", label="67 GHz")
ax.legend(); ax.grid(True)

ax = axes[1]
ax.plot(freqs * 1e-9, phase_wrapped, label="wrapped")
ax.plot(freqs * 1e-9, phase_unwrapped, label="unwrapped", ls="--", c="C1")
ax.set_ylabel("∠S21 (deg)")
ax.set_xlabel("Frequency (GHz)")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(67, color="r", ls="--")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig("s21_phase.png", dpi=150)
plt.show()

In [ ]:
data = np.load("./analysis/s21.npz", allow_pickle=True)

# --- FFT result (spectrum) ---
freqs = data["freqs"]
S21   = data["S21"]

# Wrapped phase: values stay in (-180, 180] — standard for S-params
phase_wrapped = np.degrees(np.angle(S21))

# Unwrapped phase: continuous, removes 2π jumps — useful to see
# total phase accumulation across frequency
phase_unwrapped = np.degrees(np.unwrap(np.angle(S21)))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

for ax in axes:
    ax.set_xlim(60,70)

ax = axes[0]
ax.plot(freqs * 1e-9, 20 * np.log10(np.abs(S21) + 1e-30))
ax.set_ylabel("|S21| (dB)")
ax.axvline(67, color="r", ls="--", label="67 GHz")
ax.legend(); ax.grid(True)

ax = axes[1]
ax.plot(freqs * 1e-9, phase_wrapped, label="wrapped")
ax.set_ylabel("∠S21 (deg)")
ax.set_xlabel("Frequency (GHz)")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(67, color="r", ls="--")
ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig("s21_phase_formatted.png", dpi=150)
plt.show()